# Samples, Amplicons, ecDNA, and Tumor Purity Analysis
This notebook analyzes biosample-level genomic data and generates summary plots for sample counts per patient, the distribution of amplicon and ecDNA sequence counts per independent sample, and tumor purity across tumor types and cohorts.

## Requirements
Data:
- Suppl. Tbls.

Dependencies:
- pandas,
- matplotlib,
- seaborn

## Outputs
- out/barplot_samples_per_patient
- out/barplot_amplicons_and_ecdna_per_sample
- out/tumor_purity_by_cohort
- out/tumor_purity

In [ ]:
import pandas as pd
import seaborn as sns 
import matplotlib.pyplot as plt
# Directory setup
import os
from pathlib import Path
# Import local dependencies
import sys
sys.path.append('../../src')
Path("out").mkdir(parents=True, exist_ok=True)

from data_imports import *
from seaborn_helper_functions import *

Plot samples per patient

In [ ]:
df = import_biosamples()

def plot_samples_per_patient(data):
    # reset index
    data = data.reset_index()

    # number of samples per patient
    samples_per_patient = (
        data.groupby('patient_id')['biosample_id']
        .nunique()
        .reset_index(name='sample_count')
    )

    # number of patients for each sample count
    patient_distribution = (
        samples_per_patient['sample_count']
        .value_counts()
        .sort_index()
        .reset_index()
    )
    patient_distribution.columns = ['sample_counts', 'patient_counts']

    # figure
    fig, ax = plt.subplots(figsize=(7, 2))

    sns.barplot(
        data=patient_distribution,
        x='sample_counts',
        y='patient_counts',
        ax=ax
    )

    ax.set_xlabel('Number of samples per patient')
    ax.set_ylabel('Number of patients')
    ax.set_yscale('log', base=10)

    plt.tight_layout()
    sns.despine()

    return fig, ax


fig, ax = plot_samples_per_patient(df)
savefig(fig, 'out/barplot_samples_per_patient')

Plot amplicons and ecDNA sequences per sample

In [ ]:
def plot_amplicons_and_ecdna_per_sample(df1, df2):
    '''
    df1: S.T.2 biosamples, from import_biosamples()
    df2: S.T.3 amplicons, from import_amplicons() 
    '''
    # Independent samples
    df3 = df1[df1["in_unique_patient_set"] == True].copy()

    
    # biosample_id is the index
    if "biosample_id" not in df3.columns and df3.index.name == "biosample_id":
        df3 = df3.reset_index()

    independent_samples = df3[["biosample_id"]].drop_duplicates()

    
    # Amplicons per sample
    df2 = df2.rename(columns={"sample_name": "biosample_id"})

    amplicons_per_sample = (
        df2.groupby("biosample_id")["amplicon_number"]
        .nunique()
        .reset_index(name="amplicon_count")
    )
    
    
    # Include samples with zero amplicons
    amplicons_per_sample = (
        independent_samples
        .merge(
            amplicons_per_sample,
            on="biosample_id",
            how="left"
        )
    )

    amplicons_per_sample["amplicon_count"] = (
        amplicons_per_sample["amplicon_count"]
        .fillna(0)
        .astype(int)
    )


    # ecDNA sequences per sample
    ecdna_per_sample = df3[
        ["biosample_id", "ecDNA_sequences_detected"]
    ].copy()

    ecdna_per_sample["ecDNA_sequences_detected"] = (
        pd.to_numeric(
            ecdna_per_sample["ecDNA_sequences_detected"],
            errors="coerce"
        )
        .fillna(0)
        .astype(int)
    )


    # Distribution
    amplicon_distribution = (
        amplicons_per_sample["amplicon_count"]
        .value_counts()
        .sort_index()
        .reset_index()
    )

    amplicon_distribution.columns = [
        "count",
        "sample_count"
    ]

    ecdna_distribution = (
        ecdna_per_sample["ecDNA_sequences_detected"]
        .value_counts()
        .sort_index()
        .reset_index()
    )

    ecdna_distribution.columns = [
        "count",
        "sample_count"
    ]


    # 10 or more -> 10+
    amplicon_distribution.loc[
        amplicon_distribution["count"] >= 10,
        "count"
    ] = 10

    ecdna_distribution.loc[
        ecdna_distribution["count"] >= 10,
        "count"
    ] = 10

    amplicon_distribution = (
        amplicon_distribution
        .groupby("count", as_index=False)["sample_count"]
        .sum()
    )

    ecdna_distribution = (
        ecdna_distribution
        .groupby("count", as_index=False)["sample_count"]
        .sum()
    )


    # Merge distributions
    distribution = pd.DataFrame({
        "count": range(0, 11)
    })

    distribution = distribution.merge(
        amplicon_distribution,
        on="count",
        how="left"
    )

    distribution = distribution.merge(
        ecdna_distribution,
        on="count",
        how="left",
        suffixes=("_amplicon", "_ecDNA")
    )

    distribution = distribution.fillna(0)

    distribution["count"] = distribution["count"].astype(str)

    distribution.loc[
        distribution["count"] == "10",
        "count"
    ] = "10+"

    
    # Plot
    figsize = (8, 3)

    set_plot_defaults(
        linewidth=0.75,
        fontsize=7,
        figsize=figsize
    )

    fig, ax = plt.subplots(figsize=figsize)

    x = np.arange(len(distribution))
    width = 0.4

    ax.bar(
        x - width / 2,
        distribution["sample_count_amplicon"],
        width=width,
        label="Amplicons"
    )

    ax.bar(
        x + width / 2,
        distribution["sample_count_ecDNA"],
        width=width,
        label="ecDNA sequences"
    )

    ax.set_xticks(x)
    ax.set_xticklabels(distribution["count"])

    ax.set_xlabel("Number per independent sample")
    ax.set_ylabel("Number of independent samples")

    ax.set_yscale("log", base=10)

    ax.legend(frameon=False)

    sns.despine()

    plt.tight_layout()

    return fig, ax

df1 = import_biosamples()
df2 = import_amplicons()
fig, ax = plot_amplicons_and_ecdna_per_sample(df1, df2)
savefig(fig, "out/barplot_amplicons_and_ecdna_per_sample")

Plot tumor purity for each tumor type

Plot1 stratified by cohort

In [ ]:
def plot_tumor_purity_by_cohort(df):

    # Keep only one sample per patient
    data = df[df["in_unique_patient_set"] == True].copy()

    data = data[
        ["cancer_type", "cohort", "tumor_fraction_THetA2"]
    ].copy()

    data = data.dropna(
        subset=["cancer_type", "cohort", "tumor_fraction_THetA2"]
    )

    # Classify cohorts as SJ or CBTN
    data["cohort_group"] = data["cohort"].astype(str).apply(
        lambda x: "SJ" if x.startswith("SJ") else "CBTN"
    )

    # Convert tumor fraction to percentage
    if data["tumor_fraction_THetA2"].max() <= 1:
        data["tumor_purity"] = data["tumor_fraction_THetA2"] * 100
    else:
        data["tumor_purity"] = data["tumor_fraction_THetA2"]

    # Figure
    fig, ax = plt.subplots(figsize=(24, 7))

    sns.boxplot(
        data=data,
        x="cancer_type",
        y="tumor_purity",
        hue="cohort_group",
        hue_order=["SJ", "CBTN"],
        palette={"SJ": "#009E73", "CBTN": "#CC79A7"},
        width=0.6,
        boxprops={"edgecolor": "black"},
        ax=ax
    )

    ax.set_xlabel("Tumor type")
    ax.set_ylabel("Tumor purity (%)", fontsize=20)
    ax.tick_params(axis="x", labelsize=20, rotation=90)
    ax.tick_params(axis="y", labelsize=20)
    ax.legend(fontsize=20, title_fontsize=20)

    plt.tight_layout()
    sns.despine()

    return fig, ax

fig, ax = plot_tumor_purity_by_cohort(df)
savefig(fig, "out/tumor_purity_by_cohort")

Plot 2 not stratified by cohort

In [ ]:
def plot_tumor_purity(df):

    # Keep only one sample per patient
    data = df[df["in_unique_patient_set"] == True].copy()

    data = data[
        ["cancer_type", "tumor_fraction_THetA2"]
    ].copy()

    data = data.dropna(
        subset=["cancer_type", "tumor_fraction_THetA2"]
    )

    # Convert tumor fraction to percentage
    if data["tumor_fraction_THetA2"].max() <= 1:
        data["tumor_purity"] = data["tumor_fraction_THetA2"] * 100
    else:
        data["tumor_purity"] = data["tumor_fraction_THetA2"]

    # Figure
    fig, ax = plt.subplots(figsize=(24, 7))

    sns.boxplot(
        data=data,
        x="cancer_type",
        y="tumor_purity",
        color="#0072B2",
        boxprops={"edgecolor": "black"},
        ax=ax
    )

    ax.set_xlabel("Tumor type")
    ax.set_ylabel("Tumor purity (%)", fontsize=20)
    ax.tick_params(axis="x", labelsize=20, rotation=90)
    ax.tick_params(axis="y", labelsize=20)

    plt.tight_layout()
    sns.despine()

    return fig, ax

fig, ax = plot_tumor_purity(df)
savefig(fig, "out/tumor_purity")